# YAZEL RECOVAR Integration Demo

This notebook demonstrates how RECOVAR can be used to filter false positive picks from PhaseNet.

- **RECOVAR**: Classifies waveforms to distinguish real earthquakes from noise using learned representations
- **RECOVAR YAZEL Integration**: Uses sliding windows to score PhaseNet picks and filter false positives

#### PhaseNet Configuration:
- **Overlap**: 0.90 (90% overlap between windows)
- **Stacking**: avg (average predictions across overlapping windows)
- **Model**: PhaseNet `instance` pretrained model


In [ ]:
import obspy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time
from datetime import datetime

from yazel_integration_sliding import (
    recovar_pick_cleaner_sliding,
    load_recovar_classifier
)
from demo_plotting import get_phasenet_probabilities, plot_side_by_side_comparison, plot_example
from demo_utils import load_example_picks

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

### Setup: Load Model and Data

In [2]:
# Configuration
MODEL_PATH = '/mnt/data_a/ege/recovar_models/exp_instance/representation_learning_autoencoder_ensemble/instance/split0/ep19.h5'
PHASENET_THRESHOLD = 0.32
phasenet_pick_dir = f"filtered_phasenet_picks_dir_thr_{PHASENET_THRESHOLD:.2f}"

# Load catalog for ground truth
catalog_path = '/home/boxx/Public/earthquake_model_evaluations/data/SilivriPaper_2019-09-01__2019-11-30/processed_catalogs/kara74a_phase_picks.csv'
catalog = pd.read_csv(catalog_path)
catalog['p_arrival_time'] = pd.to_datetime(catalog['p_arrival_time'])
catalog = catalog[catalog['station'] == 'SLVT']  # Filter for SLVT station only

# Load PhaseNet picks metadata
phasenet_picks = pd.read_csv(f"{phasenet_pick_dir}/metadata.csv")

print(f"Loaded {len(phasenet_picks)} PhaseNet picks")
print(f"Loaded {len(catalog)} catalog picks for station SLVT")

Loaded 2175 PhaseNet picks
Loaded 534 catalog picks for station SLVT


In [3]:
# Load RECOVAR classifier
print("Loading RECOVAR classifier...")
classifier = load_recovar_classifier(MODEL_PATH)
print("Classifier loaded successfully!")

Loading RECOVAR classifier...


2025-11-28 15:36:07.390323: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22286 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:19:00.0, compute capability: 8.6
2025-11-28 15:36:07.391350: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 15929 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:1a:00.0, compute capability: 8.6
2025-11-28 15:36:07.392196: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 16953 MB memory:  -> device: 2, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:67:00.0, compute capability: 8.6
2025-11-28 15:36:07.393843: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 654 MB memory:  -> device: 3, name: NVIDIA GeForce RTX 3090, 

Classifier loaded successfully!


## Select Example Picks

For this demonstration, we'll select:
1. **TRUE PICKS**: PhaseNet picks that match catalog events (real earthquakes)
2. **FALSE PICKS**: PhaseNet picks without catalog events (noise/artifacts)

In [ ]:
# Load and categorize picks into TRUE and FALSE examples
tp_examples, fp_examples = load_example_picks(phasenet_pick_dir, catalog, max_files=50)

print(f"Found {len(tp_examples)} TRUE PICK examples")
print(f"Found {len(fp_examples)} FALSE PICK examples")

**Note:** PhaseNet probability arrays are saved directly in the MSEED files alongside waveform data during the picking phase.

## Visualization

### TRUE PICK Examples (Real Earthquake)

## RECOVAR Filtering Examples

Demonstrating RECOVAR's filtering performance with threshold = 0.07 (max score):
- **TRUE PICK + Kept**: Real earthquake that RECOVAR correctly kept
- **TRUE PICK + Filtered**: Real earthquake that RECOVAR incorrectly rejected  
- **FALSE PICK + Filtered**: Noise/artifact that RECOVAR correctly rejected
- **FALSE PICK + Kept**: Noise/artifact that RECOVAR incorrectly kept

In [11]:
# Process all examples
tp_results = []
fp_results = []

print("Processing TRUE PICKS...")
for example in tp_examples:
    result = recovar_pick_cleaner_sliding(example['stream'], classifier)
    tp_results.append(result)

print("Processing FALSE PICKS...")
for example in fp_examples:
    result = recovar_pick_cleaner_sliding(example['stream'], classifier)
    fp_results.append(result)

# Extract scores
tp_mean_scores = [r['mean_score'] for r in tp_results]
tp_max_scores = [r['max_score'] for r in tp_results]
fp_mean_scores = [r['mean_score'] for r in fp_results]
fp_max_scores = [r['max_score'] for r in fp_results]

print(f"\nProcessed {len(tp_results)} TRUE PICKS and {len(fp_results)} FALSE PICKS")

Processing TRUE PICKS...
Processing FALSE PICKS...

Processed 3 TRUE PICKS and 46 FALSE PICKS


In [ ]:
# Find examples of each category using the threshold of 0.07
RECOVAR_THRESHOLD = 0.07

# Categorize examples
tp_kept = []  # TRUE PICK kept by RECOVAR
tp_filtered = []  # TRUE PICK filtered by RECOVAR
fp_kept = []  # FALSE PICK kept by RECOVAR
fp_filtered = []  # FALSE PICK filtered by RECOVAR

for example in tp_examples:
    result = recovar_pick_cleaner_sliding(example['stream'], classifier)
    example['recovar_result'] = result
    example['phasenet_result'] = get_phasenet_probabilities(example['stream'])
    
    if result['max_score'] >= RECOVAR_THRESHOLD:
        tp_kept.append(example)
    else:
        tp_filtered.append(example)

for example in fp_examples:
    result = recovar_pick_cleaner_sliding(example['stream'], classifier)
    example['recovar_result'] = result
    example['phasenet_result'] = get_phasenet_probabilities(example['stream'])
    
    if result['max_score'] >= RECOVAR_THRESHOLD:
        fp_kept.append(example)
    else:
        fp_filtered.append(example)

print(f"TRUE PICKS kept by RECOVAR: {len(tp_kept)}")
print(f"TRUE PICKS filtered by RECOVAR: {len(tp_filtered)}")
print(f"FALSE PICKS kept by RECOVAR: {len(fp_kept)}")
print(f"FALSE PICKS filtered by RECOVAR: {len(fp_filtered)}")

# Show multiple examples from each category
num_examples_to_show = 2

print("\n" + "="*80)
print("TRUE PICKS KEPT BY RECOVAR (Correct decisions)")
print("="*80)
for i, ex in enumerate(tp_kept[:num_examples_to_show]):
    print(f"\nExample {i+1}/{min(len(tp_kept), num_examples_to_show)}")
    plot_example(ex, ex['recovar_result'], ex['phasenet_result'], threshold=RECOVAR_THRESHOLD)

print("\n" + "="*80)
print("FALSE PICKS FILTERED BY RECOVAR (Correct decisions)")
print("="*80)
for i, ex in enumerate(fp_filtered[:num_examples_to_show]):
    print(f"\nExample {i+1}/{min(len(fp_filtered), num_examples_to_show)}")
    plot_example(ex, ex['recovar_result'], ex['phasenet_result'], threshold=RECOVAR_THRESHOLD)

if tp_filtered:
    print("\n" + "="*80)
    print("TRUE PICKS FILTERED BY RECOVAR (Incorrect decisions - False Negatives)")
    print("="*80)
    for i, ex in enumerate(tp_filtered[:num_examples_to_show]):
        print(f"\nExample {i+1}/{min(len(tp_filtered), num_examples_to_show)}")
        plot_example(ex, ex['recovar_result'], ex['phasenet_result'], threshold=RECOVAR_THRESHOLD)

if fp_kept:
    print("\n" + "="*80)
    print("FALSE PICKS KEPT BY RECOVAR (Incorrect decisions - False Positives)")
    print("="*80)
    for i, ex in enumerate(fp_kept[:num_examples_to_show]):
        print(f"\nExample {i+1}/{min(len(fp_kept), num_examples_to_show)}")
        plot_example(ex, ex['recovar_result'], ex['phasenet_result'], threshold=RECOVAR_THRESHOLD)

## Comparison: TRUE PICKS vs FALSE PICKS

Let's process multiple examples and compare the score distributions.

## Summary
- A well-chosen RECOVAR threshold can filter many false positives while retaining most true positives

For batch processing and full evaluation, see `run_yazel_batch_sliding.py`